# xGen-MM: clean-image layer selection for Exp3

Select one early/middle/deep partition from the mean **clean source + target** adjacent-layer CKA curve, then freeze it for both linear and RBF evaluation. Selection never reads attack results, adversarial caches, scores, or separation metrics. Distinct clean image contents receive equal weight, even when IDs recur in the manifest.

This notebook only reads existing caches/images/CSVs and displays results. It performs no model inference and writes no experiment outputs. Configure paths below and run from top to bottom.


In [2]:
import importlib.util
import json
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Set explicitly if the notebook kernel starts outside the AMP checkout.
PROJECT_ROOT = None
if PROJECT_ROOT is None:
    candidates = [Path.cwd(), *Path.cwd().parents, Path.cwd() / "amp"]
    PROJECT_ROOT = next(
        (path for path in candidates
         if (path / "adversarial_mislabeling_attack/xgen_mm/exp3.py").is_file()),
        None,
    )
    if PROJECT_ROOT is None:
        raise FileNotFoundError("Set PROJECT_ROOT to the AMP checkout.")
PROJECT_ROOT = Path(PROJECT_ROOT).resolve()
MODEL_DIR = PROJECT_ROOT / "adversarial_mislabeling_attack/xgen_mm"

DATA_ROOT = Path("/data/anantaraha/amp/dataset/laion_art")
MANIFEST_PATH = DATA_ROOT / "attack_set/manifest.csv"
ATTACK_RESULTS_PATH = DATA_ROOT / "attack_set/xgen_mm/attack_results.csv"
CACHE_DIR = DATA_ROOT / "attack_set/representations/xgen_mm_phi3_mini_instruct_r_v1"
EXISTING_EXP3_DIR = DATA_ROOT / "output/xgen_mm/exp3"

SELECTION_KERNEL = "linear"  # Prespecify "linear" or "rbf"; one choice, shared boundaries.
MIN_ENCODER_LAYERS = 3
MAX_SAMPLES = None  # Optional first N manifest rows, fixed before selection/evaluation.
CONDITIONS = ("source", "adversarial", "target")
KERNELS = ("linear", "rbf")
METRICS = ("A_MD", "A_DD", "S_gap")

if SELECTION_KERNEL not in KERNELS:
    raise ValueError("SELECTION_KERNEL must be linear or rbf.")
if MAX_SAMPLES is not None and (not isinstance(MAX_SAMPLES, int) or MAX_SAMPLES < 1):
    raise ValueError("MAX_SAMPLES must be a positive integer or None.")


In [3]:
def load_xgen_modules(model_dir):
    """Import the existing xGen-MM code without reusing another VLM's _vision module."""
    def load(name, path):
        spec = importlib.util.spec_from_file_location(name, path)
        module = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(module)
        return module

    old_bytecode = sys.dont_write_bytecode
    previous_vision = sys.modules.get("_vision")
    sys.dont_write_bytecode = True
    try:
        native = load("_xgen_selection_vision", model_dir / "_vision.py")
        sys.modules["_vision"] = native
        experiment = load("_xgen_selection_exp3", model_dir / "exp3.py")
    finally:
        sys.dont_write_bytecode = old_bytecode
        if previous_vision is None:
            sys.modules.pop("_vision", None)
        else:
            sys.modules["_vision"] = previous_vision
    return native, experiment

native, exp3 = load_xgen_modules(MODEL_DIR)
CKA_FUNCTIONS = {
    "linear": exp3.token_cka_matrix,
    "rbf": exp3.token_rbf_cka_matrix,
}
# The existing loader verifies model, precision, layout and image hashes.
# cache_only=True prohibits extraction and cache writes, including on invalid/missing caches.
loader = native.Representations(device="cpu", cache_dir=CACHE_DIR, cache_only=True)
print(f"Model: {native.MODEL_ID}")
print(f"Cache: {CACHE_DIR}")


Model: Salesforce/xgen-mm-phi3-mini-instruct-r-v1
Cache: /data/anantaraha/amp/dataset/laion_art/attack_set/representations/xgen_mm_phi3_mini_instruct_r_v1


## Replaceable selection rule

For `L` encoder layers, the curve has `L-1` values: `curve[i-1] = CKA(H_i, H_{i+1})`, for `i = 1, ..., L-1`. Embedding state 0 and CLS tokens are excluded from this curve.

Associate each value with its **left encoder layer**. For a cut after encoder layers `b1` and `b2`, partition the complete curve as `curve[:b1]`, `curve[b1:b2]`, and `curve[b2:]`. The encoder groups are `1..b1`, `b1+1..b2`, and `b2+1..L`; report `middle_start=b1+1` and `deep_start=b2+1`. Boundary-crossing adjacent pairs stay with their left layer. The final encoder has no outgoing adjacent pair, so three deep encoder layers contribute two curve values.

Search **every** pair satisfying `b1 >= 3`, `b2-b1 >= 3`, and `L-b2 >= 3`. Minimize the sum of squared deviations from each curve segment's mean. Exact ties choose the smallest `b1`, then `b2`. The function below accepts only the clean mean curve and encoder depth; no evaluation scores can influence it.


In [4]:
def select_three_segments(mean_curve, n_encoder_layers, min_layers=3):
    """Exhaustive fixed-three-segment SSE search under the documented left-layer convention."""
    curve = np.asarray(mean_curve, dtype=np.float64)
    if not isinstance(n_encoder_layers, (int, np.integer)):
        raise ValueError("n_encoder_layers must be an integer.")
    if not isinstance(min_layers, (int, np.integer)) or min_layers < 3:
        raise ValueError("Require at least three encoder layers per segment.")
    if n_encoder_layers < 3 * min_layers:
        raise ValueError("The encoder is too shallow for three valid segments.")
    if curve.ndim != 1 or len(curve) != n_encoder_layers - 1:
        raise ValueError("Expected one curve value for each encoder-to-encoder adjacency.")
    if not np.isfinite(curve).all():
        raise ValueError("The clean mean curve must be finite.")

    candidates = []
    for b1 in range(min_layers, n_encoder_layers - 2 * min_layers + 1):
        for b2 in range(b1 + min_layers, n_encoder_layers - min_layers + 1):
            segments = (curve[:b1], curve[b1:b2], curve[b2:])
            means = [float(segment.mean()) for segment in segments]
            sse = float(sum(np.square(segment - mean).sum()
                            for segment, mean in zip(segments, means)))
            candidates.append({
                "b1": b1, "b2": b2,
                "middle_start": b1 + 1, "deep_start": b2 + 1,
                "n_early_layers": b1, "n_middle_layers": b2 - b1,
                "n_deep_layers": n_encoder_layers - b2,
                "early_curve_mean": means[0], "middle_curve_mean": means[1],
                "deep_curve_mean": means[2], "sse": sse,
            })
    candidates = pd.DataFrame(candidates).sort_values(
        ["sse", "b1", "b2"], kind="stable"
    ).reset_index(drop=True)
    best = candidates.iloc[0].to_dict()
    for name in ["b1", "b2", "middle_start", "deep_start",
                 "n_early_layers", "n_middle_layers", "n_deep_layers"]:
        best[name] = int(best[name])
    return best, candidates


def encoder_ranges(n_encoder_layers, b1, b2):
    if not 1 <= b1 < b2 < n_encoder_layers:
        raise ValueError("Invalid encoder boundaries.")
    return {
        "early": tuple(range(1, b1 + 1)),
        "middle": tuple(range(b1 + 1, b2 + 1)),
        "deep": tuple(range(b2 + 1, n_encoder_layers + 1)),
    }


In [5]:
def build_clean_inventory(manifest):
    """Collect only source/target images, checking cache keys, IDs and content identity."""
    required = set(native.PROVENANCE) | {"source_path", "target_path"}
    missing = required - set(manifest.columns)
    if missing:
        raise ValueError(f"Missing manifest fields: {sorted(missing)}")
    if manifest["sample_id"].duplicated().any():
        raise ValueError("Duplicate sample IDs in the manifest.")
    if manifest[list(required)].apply(lambda column: column.str.strip().eq("")).any().any():
        raise ValueError("Blank manifest identifiers, concepts or paths.")

    rows, fingerprints, cache_owners, id_digests = [], {}, {}, {}
    for record in manifest.to_dict(orient="records"):
        for condition in ("source", "target"):
            image_id = record[f"{condition}_image_id"]
            filename = native.cache_name(image_id)
            if filename in cache_owners and cache_owners[filename] != image_id:
                raise ValueError(f"Clean cache key collision: {filename}")
            cache_owners[filename] = image_id
            path = native.image_path(record[f"{condition}_path"])
            if path not in fingerprints:
                fingerprints[path] = native.sha256_file(path)
            digest = fingerprints[path]
            if image_id in id_digests and id_digests[image_id] != digest:
                raise ValueError(f"Clean image ID refers to different contents: {image_id}")
            id_digests[image_id] = digest
            rows.append({
                "sample_id": record["sample_id"], "pair_id": record["pair_id"],
                "condition": condition, "image_id": image_id, "image_path": str(path),
                "image_sha256": digest, "cache_path": str(CACHE_DIR / "clean" / filename),
            })
    occurrences = pd.DataFrame(rows)
    if occurrences.empty:
        raise ValueError("No clean images in the selected manifest rows.")
    # Exact duplicate image contents receive one vote, regardless of ID or source/target role.
    unique = occurrences.drop_duplicates("image_sha256", keep="first").reset_index(drop=True)
    return occurrences, unique


def compute_clean_geometry(unique_images, cache_loader, selection_kernel):
    """Compute clean matrices once and reuse them later; return no adversarial information."""
    matrices, curves, other_kernel_errors = {}, [], []
    n_encoder_layers = None
    for index, record in enumerate(unique_images.to_dict(orient="records"), 1):
        print(f"Clean image {index}/{len(unique_images)}: {record['image_id']}", flush=True)
        try:
            payload = cache_loader.get_payload(
                Path(record["image_path"]), Path(record["cache_path"]), record["image_sha256"]
            )
            states = payload["hidden_states"]
            depth = len(states) - 1
            if n_encoder_layers is not None and depth != n_encoder_layers:
                raise ValueError("Clean caches have different encoder depths.")
            n_encoder_layers = depth
            # Compute the prespecified selection kernel first. Missing/undefined selection
            # data stops the notebook rather than silently changing the clean cohort.
            selected_matrix = CKA_FUNCTIONS[selection_kernel](states)
        except Exception as error:
            raise RuntimeError(
                f"Cannot form the clean selection curve for {record['image_id']}: {error}. "
                "No cache will be generated or replaced."
            ) from error
        image_matrices = {selection_kernel: selected_matrix}
        for kernel in KERNELS:
            if kernel == selection_kernel:
                continue
            try:
                image_matrices[kernel] = CKA_FUNCTIONS[kernel](states)
            except Exception as error:
                image_matrices[kernel] = None
                other_kernel_errors.append({
                    "image_id": record["image_id"], "image_sha256": record["image_sha256"],
                    "cka": kernel, "reason": f"{type(error).__name__}: {error}",
                })
        matrices[record["image_sha256"]] = image_matrices
        # Full matrices retain Exp3 indexing; use only encoder pairs (1,2),...,(L-1,L).
        curve = selected_matrix[np.arange(1, depth), np.arange(2, depth + 1)]
        curves.append(curve)
        del payload, states
    return n_encoder_layers, np.stack(curves), matrices, pd.DataFrame(other_kernel_errors)


In [6]:
manifest_digest = native.sha256_file(MANIFEST_PATH)
manifest = pd.read_csv(MANIFEST_PATH, dtype=str, keep_default_na=False)
if MAX_SAMPLES is not None:
    manifest = manifest.head(MAX_SAMPLES).copy()

clean_occurrences, unique_clean_images = build_clean_inventory(manifest)
print(f"Manifest samples: {len(manifest)}")
print(f"Clean occurrences: {len(clean_occurrences)}; distinct clean image contents: {len(unique_clean_images)}")
n_encoder_layers, clean_curves, clean_matrices, clean_kernel_errors = compute_clean_geometry(
    unique_clean_images, loader, SELECTION_KERNEL
)
mean_clean_curve = clean_curves.mean(axis=0)
curve_summary = pd.DataFrame({
    "left_layer": np.arange(1, n_encoder_layers),
    "right_layer": np.arange(2, n_encoder_layers + 1),
    "mean_cka": mean_clean_curve,
    "std_across_clean_images": pd.DataFrame(clean_curves).std(axis=0, ddof=1).to_numpy(),
    "n_unique_clean_images": len(unique_clean_images),
})
display(curve_summary)
if not clean_kernel_errors.empty:
    print("The non-selection kernel has undefined clean matrices; affected evaluation triplets will be reported.")
    display(clean_kernel_errors)


Manifest samples: 50
Clean occurrences: 100; distinct clean image contents: 100
Clean image 1/100: 000000036100
Clean image 2/100: 000000079519
Clean image 3/100: 000000073802
Clean image 4/100: 000000062068
Clean image 5/100: 000000028060


KeyboardInterrupt: 

In [ ]:
selection, selection_candidates = select_three_segments(
    mean_clean_curve, n_encoder_layers, MIN_ENCODER_LAYERS
)
selected_boundaries = (selection["b1"], selection["b2"])
selected_ranges = encoder_ranges(n_encoder_layers, *selected_boundaries)

# Exact equal-thirds convention from the existing xGen-MM Exp3 implementation.
equal_blocks, equal_middle, equal_deep = exp3.layer_blocks(n_encoder_layers + 1)
equal_ranges = {
    name: tuple(layer for layer, block in equal_blocks.items() if block == name)
    for name in ("early", "middle", "deep")
}
GROUPINGS = {"equal_thirds": equal_ranges, "clean_selected": selected_ranges}
selection_record = {
    "selection_kernel": SELECTION_KERNEL,
    "manifest_sha256": manifest_digest,
    "n_unique_clean_images": len(unique_clean_images),
    "encoder_layers": n_encoder_layers,
    "minimum_encoder_layers_per_segment": MIN_ENCODER_LAYERS,
    "b1": selection["b1"], "b2": selection["b2"],
    "middle_start": selection["middle_start"], "deep_start": selection["deep_start"],
    "objective_sse": selection["sse"],
    "valid_candidates": len(selection_candidates),
    "tie_break": "smallest b1, then smallest b2",
}
print(json.dumps(selection_record, indent=2))
range_table = pd.DataFrame([
    {"layer_ranges": name, "early": f"{groups['early'][0]}..{groups['early'][-1]}",
     "middle": f"{groups['middle'][0]}..{groups['middle'][-1]}",
     "deep": f"{groups['deep'][0]}..{groups['deep'][-1]}",
     "middle_start": groups["middle"][0], "deep_start": groups["deep"][0]}
    for name, groups in GROUPINGS.items()
])
display(range_table)
display(selection_candidates.head(10))


In [ ]:
x = curve_summary["left_layer"].to_numpy()
mean = curve_summary["mean_cka"].to_numpy()
spread = curve_summary["std_across_clean_images"].fillna(0).to_numpy()
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(x, mean, marker="o", label=f"Mean clean {SELECTION_KERNEL.upper()} CKA")
ax.fill_between(x, mean - spread, mean + spread, alpha=0.15, label="±1 image SD (not a confidence interval)")
b1, b2 = selected_boundaries
for label, lower, upper, segment, color in [
    ("early", 0.5, b1 + 0.5, mean[:b1], "tab:blue"),
    ("middle", b1 + 0.5, b2 + 0.5, mean[b1:b2], "tab:orange"),
    ("deep", b2 + 0.5, n_encoder_layers + 0.5, mean[b2:], "tab:green"),
]:
    ax.axvspan(lower, upper, color=color, alpha=0.07)
    ax.hlines(segment.mean(), lower, upper, color=color, linestyle=":", label=f"{label} segment mean")
for boundary, name in [(b1, "middle_start"), (b2, "deep_start")]:
    ax.axvline(boundary + 0.5, color="black", linestyle="--", linewidth=1)
    ax.text(boundary + 0.5, 0.98, f" {name}={boundary + 1}",
            transform=ax.get_xaxis_transform(), va="top", rotation=90)
ax.set_xlim(0.5, n_encoder_layers + 0.5)
ax.set_xlabel(r"Left encoder layer i; curve value is CKA($H_i$, $H_{i+1}$)")
ax.set_ylabel("Mean adjacent-layer patch-token CKA")
ax.set_title(f"xGen-MM clean-only selection: {len(unique_clean_images)} distinct images; SSE={selection['sse']:.6g}")
ax.legend(fontsize=8, loc="best")
plt.tight_layout()
plt.show()


## Frozen-range Exp3 evaluation

Only now read attack results and adversarial caches. Reuse the existing Exp3 linear/RBF CKA functions and `gap_scores`, including CLS removal, RBF bandwidths, unique off-diagonal deep-layer pairs, and `S_gap = A_DD - A_MD`. Both kernels use the **same frozen clean-selected ranges** and the same equal-thirds reference.

Every admitted sample contributes source/adversarial/target together. Cache/provenance failures or undefined kernels are reported. Comparisons below use the common complete sample set across both kernels and both range definitions. Clean geometry is reused only for byte-identical images verified by SHA-256.


In [ ]:
def evaluate_frozen_ranges(selected_manifest, occurrences, unique_images, matrices, cache_loader, groupings):
    if native.sha256_file(MANIFEST_PATH) != manifest_digest:
        raise ValueError("Manifest changed after clean-only selection; restart the notebook.")
    _, samples = native.read_samples(MANIFEST_PATH, ATTACK_RESULTS_PATH)
    samples = samples.loc[samples["sample_id"].isin(selected_manifest["sample_id"])]
    clean_lookup = occurrences.set_index(["sample_id", "condition"])
    canonical = unique_images.set_index("image_sha256")
    fingerprints, rows, errors = {}, [], []
    for index, record in enumerate(samples.to_dict(orient="records"), 1):
        sample_id = record["sample_id"]
        print(f"Evaluation {index}/{len(samples)}: {sample_id}", flush=True)
        status = str(record.get("_attack_status", "")).strip().lower()
        if status not in native.SUCCESS_STATUSES:
            errors.append({"sample_id": sample_id, "cka": "both", "reason": f"attack status={status!r}"})
            continue
        try:
            paths = native.validate_sample(record, fingerprints)
            clean_records = {condition: clean_lookup.loc[(sample_id, condition)]
                             for condition in ("source", "target")}
            for condition, clean_record in clean_records.items():
                if fingerprints[paths[condition]] != clean_record["image_sha256"]:
                    raise ValueError(f"{condition} image changed after clean selection")
            adv_cache = CACHE_DIR / "adv" / native.cache_name(sample_id)
            payload = cache_loader.get_payload(paths["adversarial"], adv_cache,
                                               fingerprints[paths["adversarial"]])
            states = payload["hidden_states"]
            if len(states) - 1 != n_encoder_layers:
                raise ValueError("Adversarial encoder depth differs from the frozen clean depth.")
        except Exception as error:
            errors.append({"sample_id": sample_id, "cka": "both",
                           "reason": f"{type(error).__name__}: {error}"})
            continue

        for kernel in KERNELS:
            try:
                sample_matrices = {
                    condition: matrices[clean_record["image_sha256"]][kernel]
                    for condition, clean_record in clean_records.items()
                }
                if any(matrix is None for matrix in sample_matrices.values()):
                    raise ValueError("Undefined clean CKA for this kernel; see clean_kernel_errors.")
                sample_matrices["adversarial"] = CKA_FUNCTIONS[kernel](states)
                sample_rows = []
                for grouping_name, groups in groupings.items():
                    for condition in CONDITIONS:
                        digest = fingerprints[paths[condition]]
                        used_cache = (adv_cache if condition == "adversarial"
                                      else Path(canonical.loc[digest, "cache_path"]))
                        sample_rows.append({
                            **{column: record[column] for column in native.PROVENANCE},
                            "condition": condition,
                            "image_id": sample_id if condition == "adversarial" else record[f"{condition}_image_id"],
                            "image_path": str(paths[condition]), "image_sha256": digest,
                            "cache_path": str(used_cache), "cache_validation": "sha256_verified",
                            "cka": kernel, "layer_ranges": grouping_name,
                            "middle_start": groups["middle"][0], "deep_start": groups["deep"][0],
                            "n_layers": len(states), "n_patch_tokens": states[0].shape[1] - native.PREFIX_TOKENS,
                            **exp3.gap_scores(sample_matrices[condition], groups["middle"], groups["deep"]),
                        })
                rows.extend(sample_rows)
            except Exception as error:
                errors.append({"sample_id": sample_id, "cka": kernel,
                               "reason": f"{type(error).__name__}: {error}"})
        del payload, states
    return pd.DataFrame(rows), pd.DataFrame(errors, columns=["sample_id", "cka", "reason"])


In [ ]:
image_scores, skipped_evaluations = evaluate_frozen_ranges(
    manifest, clean_occurrences, unique_clean_images, clean_matrices, loader, GROUPINGS
)
if not skipped_evaluations.empty:
    display(skipped_evaluations)
if image_scores.empty:
    raise RuntimeError("No valid frozen-range evaluation triplets; inspect skipped_evaluations.")
if image_scores.duplicated(["cka", "layer_ranges", "sample_id", "condition"]).any():
    raise ValueError("Duplicate evaluation rows.")
print(f"Cache operations: {loader.cache_counts}; extraction must remain zero.")
assert loader.cache_counts["extracted"] == 0
display(image_scores.groupby(["cka", "layer_ranges", "condition"]).size().rename("n_rows").to_frame())


In [ ]:
# Optional reproducibility check. Existing score CSVs never participate in layer selection.
existing_checks = []
for kernel in KERNELS:
    filename = "exp3_image_scores.csv" if kernel == "linear" else "exp3_rbf_image_scores.csv"
    path = EXISTING_EXP3_DIR / filename
    if not path.is_file():
        existing_checks.append({"cka": kernel, "existing_csv": str(path), "status": "not present"})
        continue
    existing = pd.read_csv(path, dtype={column: str for column in native.PROVENANCE})
    recomputed = image_scores.loc[
        (image_scores["cka"] == kernel) & (image_scores["layer_ranges"] == "equal_thirds")
    ]
    matched = recomputed.merge(existing, on=["sample_id", "condition"],
                               suffixes=("_new", "_existing"), validate="one_to_one")
    for column in native.PROVENANCE[1:]:
        if not matched[f"{column}_new"].fillna("").equals(matched[f"{column}_existing"].fillna("")):
            raise ValueError(f"Existing {kernel} results have mismatched {column}.")
    record = {"cka": kernel, "existing_csv": str(path), "n_matching_image_rows": len(matched)}
    for metric in METRICS:
        differences = (matched[f"{metric}_new"] - matched[f"{metric}_existing"]).abs()
        record[f"max_abs_{metric}_difference"] = differences.max()
    record["status"] = ("no common rows" if matched.empty else
                        "matches" if all(np.allclose(matched[f"{metric}_new"], matched[f"{metric}_existing"],
                                                     rtol=1e-10, atol=1e-12) for metric in METRICS)
                        else "differs: check run/cache provenance")
    existing_checks.append(record)
display(pd.DataFrame(existing_checks))


In [ ]:
variants = [(kernel, grouping) for kernel in KERNELS for grouping in GROUPINGS]
complete_ids = {}
for kernel, grouping in variants:
    frame = image_scores.loc[(image_scores["cka"] == kernel) & (image_scores["layer_ranges"] == grouping)]
    complete = frame.groupby("sample_id")["condition"].agg(
        lambda values: set(values) == set(CONDITIONS) and len(values) == len(CONDITIONS)
    )
    complete_ids[(kernel, grouping)] = set(complete.index[complete])
common_ids = set.intersection(*complete_ids.values())
if not common_ids:
    raise RuntimeError("No common complete triplets across both kernels and both range definitions.")
comparison_scores = image_scores.loc[image_scores["sample_id"].isin(common_ids)].copy()
if not np.isfinite(comparison_scores[list(METRICS)].to_numpy(dtype=float)).all():
    raise ValueError("Nonfinite comparison scores.")
print(f"Matched comparison: {len(common_ids)} samples out of {len(manifest)} selected manifest samples.")
display(pd.DataFrame([
    {"cka": kernel, "layer_ranges": grouping, "available_complete_samples": len(ids),
     "comparison_samples": len(common_ids), "excluded_from_comparison": len(ids - common_ids)}
    for (kernel, grouping), ids in complete_ids.items()
]))


In [ ]:
condition_summary = (
    comparison_scores.groupby(["cka", "layer_ranges", "condition"])[list(METRICS)]
    .agg(["mean", "median", "std"])
    .reindex(pd.MultiIndex.from_product(
        [KERNELS, tuple(GROUPINGS), CONDITIONS], names=["cka", "layer_ranges", "condition"]
    ))
)
condition_summary.columns = [f"{metric}_{stat}" for metric, stat in condition_summary.columns]
condition_counts = comparison_scores.groupby(["cka", "layer_ranges", "condition"]).agg(
    n_samples=("sample_id", "nunique"), n_unique_image_contents=("image_sha256", "nunique")
)
display(condition_summary.join(condition_counts).round(4))


In [ ]:
paired_scores, paired_rows = {}, []
for kernel, grouping in variants:
    frame = comparison_scores.loc[
        (comparison_scores["cka"] == kernel) & (comparison_scores["layer_ranges"] == grouping)
    ]
    pairs = frame.pivot(index="sample_id", columns="condition", values="S_gap")
    paired_scores[(kernel, grouping)] = pairs
    delta = pairs["adversarial"] - pairs["source"]
    above = int((delta > 0).sum())
    paired_rows.append({
        "cka": kernel, "layer_ranges": grouping, "n_samples": len(pairs),
        "n_concept_pairs": frame["pair_id"].nunique(),
        "adv_above_source": above, "fraction_adv_above_source": above / len(pairs),
        "mean_adv_minus_source": delta.mean(), "median_adv_minus_source": delta.median(),
        "std_adv_minus_source": delta.std(ddof=1),
    })
paired_summary = pd.DataFrame(paired_rows).set_index(["cka", "layer_ranges"])
display(paired_summary.round(4))


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for row_index, kernel in enumerate(KERNELS):
    values = comparison_scores.loc[comparison_scores["cka"] == kernel, "S_gap"]
    low, high = values.min(), values.max()
    if low == high:
        low, high = low - 1e-6, high + 1e-6
    for column_index, grouping in enumerate(GROUPINGS):
        ax = axes[row_index, column_index]
        pairs = paired_scores[(kernel, grouping)]
        ax.scatter(pairs["source"], pairs["adversarial"], s=45, alpha=0.8)
        ax.plot([low, high], [low, high], "--", linewidth=1.5)
        ax.set_xlim(low, high); ax.set_ylim(low, high)
        ax.set_xlabel(r"Source $S_{gap}$"); ax.set_ylabel(r"Adversarial $S_{gap}$")
        ax.set_title(f"{kernel.upper()} — {grouping}")
        above = int((pairs["adversarial"] > pairs["source"]).sum())
        ax.text(0.05, 0.95, f"{above}/{len(pairs)} adversarial samples above source",
                transform=ax.transAxes, va="top")
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharey=True)
for row_index, kernel in enumerate(KERNELS):
    values = comparison_scores.loc[comparison_scores["cka"] == kernel, "S_gap"]
    low, high = values.min(), values.max()
    if low == high:
        low, high = low - 1e-6, high + 1e-6
    bins = np.linspace(low, high, 20)  # Shared across range definitions for this kernel.
    for column_index, grouping in enumerate(GROUPINGS):
        ax = axes[row_index, column_index]
        frame = comparison_scores.loc[
            (comparison_scores["cka"] == kernel) & (comparison_scores["layer_ranges"] == grouping)
        ]
        for condition in CONDITIONS:
            scores = frame.loc[frame["condition"] == condition, "S_gap"]
            ax.hist(scores, bins=bins, alpha=0.5, label=f"{condition.capitalize()} (n={len(scores)})")
        ax.set_xlabel(r"$S_{gap} = A_{DD} - A_{MD}$")
        ax.set_ylabel("Number of image occurrences")
        ax.set_title(f"{kernel.upper()} — {grouping}")
        ax.legend()
plt.tight_layout()
plt.show()


## Exploratory AUROC comparison

The current reference notebook has paired/distribution summaries but no AUROC implementation. Add standard empirical AUROC here with **adversarial = positive** and **larger `S_gap` = more positive**, fixed in advance. Do not reverse the score direction, maximize AUROC over partitions, or choose a detection threshold.

Report adversarial versus source, target, and pooled source+target. Like the Exp3 summaries, these metrics weight manifest image occurrences; repeated clean images are correlated. This is an in-sample exploratory comparison, not held-out detection performance: the clean selection images overlap the clean evaluation images.


In [ ]:
def empirical_auroc(negative_scores, positive_scores):
    """P(positive > negative) + 0.5 P(tie), computed using average ranks."""
    negative = np.asarray(negative_scores, dtype=np.float64)
    positive = np.asarray(positive_scores, dtype=np.float64)
    if negative.ndim != 1 or positive.ndim != 1 or not len(negative) or not len(positive):
        raise ValueError("AUROC requires nonempty one-dimensional positive and negative scores.")
    combined = np.concatenate([negative, positive])
    if not np.isfinite(combined).all():
        raise ValueError("AUROC scores must be finite.")
    ranks = pd.Series(combined).rank(method="average").to_numpy()
    n_positive, n_negative = len(positive), len(negative)
    return float((ranks[n_negative:].sum() - n_positive * (n_positive + 1) / 2)
                 / (n_positive * n_negative))


auroc_rows = []
negative_sets = {"source": ("source",), "target": ("target",), "source+target": ("source", "target")}
for kernel, grouping in variants:
    frame = comparison_scores.loc[
        (comparison_scores["cka"] == kernel) & (comparison_scores["layer_ranges"] == grouping)
    ]
    positive = frame.loc[frame["condition"] == "adversarial"]
    for name, conditions in negative_sets.items():
        negative = frame.loc[frame["condition"].isin(conditions)]
        auroc_rows.append({
            "cka": kernel, "layer_ranges": grouping, "negative_condition": name,
            "n_positive_occurrences": len(positive), "n_negative_occurrences": len(negative),
            "n_unique_negative_images": negative["image_sha256"].nunique(),
            "AUROC": empirical_auroc(negative["S_gap"], positive["S_gap"]),
        })
auroc_summary = pd.DataFrame(auroc_rows)
display(auroc_summary.round(4))

auroc_comparison = auroc_summary.pivot(
    index=["cka", "negative_condition"], columns="layer_ranges", values="AUROC"
)
auroc_comparison["selected_minus_thirds"] = (
    auroc_comparison["clean_selected"] - auroc_comparison["equal_thirds"]
)
display(auroc_comparison.round(4))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, kernel in zip(axes, KERNELS):
    table = auroc_comparison.loc[kernel].reindex(list(negative_sets))
    x = np.arange(len(table))
    for index, grouping in enumerate(GROUPINGS):
        ax.bar(x + (index - 0.5) * 0.35, table[grouping], width=0.35, label=grouping)
    ax.axhline(0.5, color="black", linestyle="--", linewidth=1)
    ax.set_xticks(x, labels=table.index)
    ax.set_ylim(0, 1)
    ax.set_xlabel("Negative condition (adversarial is positive)")
    ax.set_ylabel(r"Empirical AUROC using $S_{gap}$")
    ax.set_title(f"xGen-MM {kernel.upper()}")
    ax.legend()
plt.tight_layout()
plt.show()


The selected starts and candidate SSEs describe a clean-representation depth partition. The paired, distribution and AUROC tables describe what happens **after freezing it**; they do not feed back into selection. Keep the signed AUROC difference, including negative differences, and the prespecified score direction.

A larger `S_gap` means deep layers align more strongly with one another than with middle layers. Improvements on this sample set would motivate independent validation, not establish a detector. To try a different selection rule later, replace `select_three_segments` while retaining the clean-only inputs, explicit layer indexing and fixed downstream evaluation.
